# DoRA for AIME — Colab Notebook
**CS 4782 · Cornell · Spring 2026** &nbsp;·&nbsp; Ryan Ye · Boaz Ng · Leon Jiao · Aadi Singla

Reproducing *DoRA: Weight-Decomposed Low-Rank Adaptation* (Liu et al., ICML 2024) on AIME competition math.

**Quick-start:**
1. Run **§ 0 Setup** (install, GPU, HF auth)
2. Edit **`CFG`** in § 1 for your experiment
3. Run a single experiment cell in **§ 4**
4. Compare everything in **§ 5 Results**

---
| Section | Contents |
|---------|----------|
| § 0 | Install · GPU check · HF auth |
| § 1 | `CFG` — one place for all settings |
| § 2 | Helper functions (train / eval / results) |
| § 3 | Data stats & preview |
| § 4 | Experiments: E3 zero-shot · E1 LoRA · E2 peft-DoRA · E4 rank-sweep · E5 QDoRA |
| § 5 | Results comparison table |

---
## § 0 — Setup

In [ ]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = '/content/cs4782-project'
    if not os.path.exists(REPO):
        print('Cloning repo ...')
        os.system('git clone https://github.com/leonjiao5/cs4782-project.git ' + REPO)
    os.chdir(REPO)
else:
    # Local: run from project root
    pass

ROOT = os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
print('CWD:', ROOT)

In [ ]:
# Install dependencies (once per Colab session)
# Uncomment the first time:
# !pip install -q -r requirements.txt

import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    gpu_info = result.stdout.strip()
    vram_gb = float(gpu_info.split(',')[1].replace(' MiB','').strip()) / 1024
    print(f'GPU: {gpu_info}')
    print(f'  => use load_in_4bit=True if VRAM <= 16 GB (T4)  |  bf16 ok above 20 GB')
else:
    print('No GPU detected. Training will be very slow.')

In [ ]:
# HuggingFace login (required for gated Llama models)
# Store your token in Colab Secrets as HF_TOKEN, or set the env var locally.
HF_TOKEN = None
if IN_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or os.environ.get('HF_TOKEN', '')

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('Logged in to HuggingFace.')
else:
    print('WARNING: HF_TOKEN not set. Gated models (Llama-3.1) will fail.')
    print('  Colab: Secrets sidebar -> add HF_TOKEN')
    print('  Local: export HF_TOKEN=hf_...')

In [ ]:
# Core imports (reload after code edits without restarting runtime)
import importlib, json, glob
from argparse import Namespace
from pathlib import Path

import yaml
import pandas as pd

import code.train as _train_mod
import code.eval  as _eval_mod

print('Imports OK')

---
## § 1 — Config

**Edit `CFG` here.** All experiment cells read from this dict.  
`apply_config()` writes it to `code/configs/run_config.yaml` before each run.

In [ ]:
CFG = {
    # ── Model ──────────────────────────────────────────────────
    'model_name'  : 'meta-llama/Llama-3.2-3B-Instruct',  # ungated, fits T4 in bf16
    'load_in_4bit': False,   # 3B fits without quantisation; set True if VRAM < 10 GB

    # ── LoRA / DoRA (paper Table 8) ────────────────────────────
    'rank'           : 16,
    'alpha'          : 32,   # always 2 * rank in the paper
    'dropout'        : 0.05,
    'target_modules' : ['q_proj', 'k_proj', 'v_proj', 'up_proj', 'down_proj'],

    # ── Training ───────────────────────────────────────────────
    'lr'                   : 2e-4,
    'epochs'               : 3,
    'batch_size'           : 16,     # effective (grad_accum = batch_size / per_device)
    'per_device_batch_size': 1,
    'warmup_ratio'         : 0.03,
    'max_seq_len'          : 4096,
    'seed'                 : 42,
    'tier'                 : 'train',   # train_light (7.5K) | train (27.5K)

    # ── Checkpointing ──────────────────────────────────────────
    'save_strategy'    : 'steps',
    'save_steps'       : 200,    # save every 200 steps (~every 15-20 min on T4)
    'save_total_limit' : 5,      # keep only the 5 most recent mid-run checkpoints

    # ── Paths (Drive setup cell below overrides these on Colab) ─
    'output_dir'  : 'results/checkpoints',
}

In [ ]:
# ── Google Drive: persist checkpoints and results across sessions ──────────
# Run this cell every time you open a new Colab session. Safe to skip locally.

if IN_COLAB:
    import shutil, glob as _glob

    DRIVE_DIR = '/content/drive/MyDrive/cs4782-dora'
    os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
    os.makedirs(f'{DRIVE_DIR}/results/tables', exist_ok=True)
    os.makedirs(f'{DRIVE_DIR}/results/logs', exist_ok=True)

    # Symlink REPO/results → Drive so eval.py writes there automatically
    results_local = f'{REPO}/results'
    if os.path.isdir(results_local) and not os.path.islink(results_local):
        # Preserve any results already written to the local dir
        for f in _glob.glob(f'{results_local}/**/*', recursive=True):
            if os.path.isfile(f):
                dst = f.replace(results_local, f'{DRIVE_DIR}/results', 1)
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                if not os.path.exists(dst):
                    shutil.copy2(f, dst)
        shutil.rmtree(results_local)
    if not os.path.exists(results_local):
        os.symlink(f'{DRIVE_DIR}/results', results_local)

    CFG['output_dir'] = f'{DRIVE_DIR}/checkpoints'
    print(f'Checkpoints → {CFG["output_dir"]}')
    print(f'Results     → {os.path.realpath(results_local)}')
else:
    print('Not on Colab — using local paths from CFG.')

# ─────────────────────────────────────────────────────────────────────────────
# Config
# ─────────────────────────────────────────────────────────────────────────────

def apply_config(overrides=None, path='code/configs/run_config.yaml'):
    """Merge CFG + overrides into a YAML file that train/eval scripts read."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    merged = {**CFG, **(overrides or {})}
    with open(path, 'w') as f:
        yaml.dump(merged, f, default_flow_style=False)
    return path


# ─────────────────────────────────────────────────────────────────────────────
# Training
# ─────────────────────────────────────────────────────────────────────────────

def run_train(method, tier=None, overrides=None, resume=None):
    """
    Train a model adapter.

    method    : 'lora' | 'peft_dora' | 'dora'
    tier      : 'train_light' | 'train' | 'train_scale'  (default: CFG['tier'])
    overrides : dict of CFG keys to change for this run only (e.g. {'rank': 8})
    resume    : path to a Trainer checkpoint dir to resume from (e.g.
                '/content/drive/MyDrive/cs4782-dora/checkpoints/lora_train_0501_2230/checkpoint-400')
    """
    config_path = apply_config(overrides)
    importlib.reload(_train_mod)
    args = Namespace(
        config=config_path,
        model=None,
        method=method,
        tier=tier or CFG['tier'],
        output_dir=CFG['output_dir'],
        resume=resume,
    )
    rank_val = (overrides or {}).get('rank', CFG['rank'])
    print(f'[TRAIN] method={method}  tier={args.tier}  rank={rank_val}'
          + (f'  resume={resume}' if resume else ''))
    _train_mod.main(args)


# ─────────────────────────────────────────────────────────────────────────────
# Evaluation
# ─────────────────────────────────────────────────────────────────────────────

def run_eval(checkpoint, benchmarks=('aime2024', 'aime2025'),
             n_samples=1, temperature=0.0, output=None):
    """
    Evaluate a checkpoint on AIME.

    checkpoint : local peft/dora dir  OR  HF model name (zero-shot)
    benchmarks : 'aime2024' | 'aime2025' | ('aime2024', 'aime2025')
    n_samples  : 1 = greedy;  >1 = majority-vote at given temperature
    """
    importlib.reload(_eval_mod)
    if isinstance(benchmarks, str):
        benchmarks = (benchmarks,)
    for bench in benchmarks:
        args = Namespace(
            checkpoint=checkpoint,
            benchmark=bench,
            n_samples=n_samples,
            temperature=temperature,
            model=None,
            load_in_4bit=CFG.get('load_in_4bit', False),
            output=output,
        )
        name = Path(checkpoint).name if Path(checkpoint).exists() else checkpoint
        print(f'[EVAL]  {name}  /  {bench}')
        _eval_mod.main(args)


# ─────────────────────────────────────────────────────────────────────────────
# Checkpoint + results inspection
# ─────────────────────────────────────────────────────────────────────────────

def list_checkpoints(output_dir=None):
    """Print a table of all saved checkpoints and return their paths."""
    d = output_dir or CFG['output_dir']
    rows = []
    for ckpt in sorted(glob.glob(os.path.join(d, '*'))):
        rc_path = os.path.join(ckpt, 'run_config.json')
        rc = json.load(open(rc_path)) if os.path.exists(rc_path) else {}
        rows.append({
            'path'  : ckpt,
            'name'  : os.path.basename(ckpt),
            'method': rc.get('method', '?'),
            'tier'  : rc.get('tier', '?'),
            'rank'  : rc.get('rank', '?'),
        })
    if rows:
        display(pd.DataFrame(rows).drop(columns=['path']))
    else:
        print('No checkpoints yet in', d)
    return [r['path'] for r in rows]


def latest_checkpoint(method_substr):
    """Return the most recent checkpoint whose name contains method_substr."""
    ckpts = sorted(glob.glob(os.path.join(CFG['output_dir'], f'*{method_substr}*')))
    if not ckpts:
        raise FileNotFoundError(f'No checkpoint matching *{method_substr}* in {CFG["output_dir"]}')
    return ckpts[-1]


def show_results(results_dir=None):
    """Load all eval JSON files and display a sorted comparison table."""
    d = results_dir or os.path.join('results', 'tables')
    rows = []
    for p in sorted(glob.glob(os.path.join(d, '*.json'))):
        r = json.load(open(p))
        rows.append({
            'checkpoint': Path(r.get('checkpoint', '')).name,
            'benchmark' : r.get('benchmark', ''),
            'accuracy'  : f"{r.get('accuracy', 0):.1%}",
            'correct'   : f"{r.get('n_correct', 0)}/{r.get('n_total', 0)}",
            'n_samples' : r.get('n_samples', 1),
        })
    if rows:
        df = (pd.DataFrame(rows)
                .sort_values(['benchmark', 'accuracy'], ascending=[True, False])
                .reset_index(drop=True))
        display(df)
    else:
        print('No results yet — run evaluation cells first.')
    return rows


print('Helpers loaded.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Config
# ─────────────────────────────────────────────────────────────────────────────

def apply_config(overrides=None, path='code/configs/run_config.yaml'):
    """Merge CFG + overrides into a YAML file that train/eval scripts read."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    merged = {**CFG, **(overrides or {})}
    with open(path, 'w') as f:
        yaml.dump(merged, f, default_flow_style=False)
    return path


# ─────────────────────────────────────────────────────────────────────────────
# Training
# ─────────────────────────────────────────────────────────────────────────────

def run_train(method, tier=None, overrides=None):
    """
    Train a model adapter.

    method    : 'lora' | 'peft_dora' | 'dora'
    tier      : 'train_light' | 'train' | 'train_scale'  (default: CFG['tier'])
    overrides : dict of CFG keys to change for this run only (e.g. {'rank': 8})
    """
    config_path = apply_config(overrides)
    importlib.reload(_train_mod)
    args = Namespace(
        config=config_path,
        model=None,           # model_name already written to run_config.yaml
        method=method,
        tier=tier or CFG['tier'],
        output_dir=CFG['output_dir'],
    )
    print(f'[TRAIN] method={method}  tier={args.tier}  rank={CFG["rank"] if not overrides else overrides.get("rank", CFG["rank"])}')
    _train_mod.main(args)


# ─────────────────────────────────────────────────────────────────────────────
# Evaluation
# ─────────────────────────────────────────────────────────────────────────────

def run_eval(checkpoint, benchmarks=('aime2024', 'aime2025'),
             n_samples=1, temperature=0.0, output=None):
    """
    Evaluate a checkpoint on AIME.

    checkpoint : local peft dir  OR  HF model name (zero-shot)
    benchmarks : 'aime2024' | 'aime2025' | ('aime2024', 'aime2025')
    n_samples  : 1 = greedy;  >1 = majority-vote at given temperature
    """
    importlib.reload(_eval_mod)
    if isinstance(benchmarks, str):
        benchmarks = (benchmarks,)
    for bench in benchmarks:
        args = Namespace(
            checkpoint=checkpoint,
            benchmark=bench,
            n_samples=n_samples,
            temperature=temperature,
            model=None,
            load_in_4bit=CFG.get('load_in_4bit', False),
            output=output,
        )
        name = Path(checkpoint).name if Path(checkpoint).exists() else checkpoint
        print(f'[EVAL]  {name}  /  {bench}')
        _eval_mod.main(args)


# ─────────────────────────────────────────────────────────────────────────────
# Checkpoint + results inspection
# ─────────────────────────────────────────────────────────────────────────────

def list_checkpoints(output_dir=None):
    """Print a table of all saved checkpoints and return their paths."""
    d = output_dir or CFG['output_dir']
    rows = []
    for ckpt in sorted(glob.glob(os.path.join(d, '*'))):
        rc_path = os.path.join(ckpt, 'run_config.json')
        rc = json.load(open(rc_path)) if os.path.exists(rc_path) else {}
        rows.append({
            'path'  : ckpt,
            'name'  : os.path.basename(ckpt),
            'method': rc.get('method', '?'),
            'tier'  : rc.get('tier', '?'),
            'rank'  : rc.get('rank', '?'),
        })
    if rows:
        display(pd.DataFrame(rows).drop(columns=['path']))
    else:
        print('No checkpoints yet in', d)
    return [r['path'] for r in rows]


def latest_checkpoint(method_substr):
    """Return the most recent checkpoint whose name contains method_substr."""
    ckpts = sorted(glob.glob(os.path.join(CFG['output_dir'], f'*{method_substr}*')))
    if not ckpts:
        raise FileNotFoundError(f'No checkpoint matching *{method_substr}* in {CFG["output_dir"]}')
    return ckpts[-1]


def show_results(results_dir=None):
    """Load all eval JSON files and display a sorted comparison table."""
    d = results_dir or os.path.join('results', 'tables')
    rows = []
    for p in sorted(glob.glob(os.path.join(d, '*.json'))):
        r = json.load(open(p))
        rows.append({
            'checkpoint': Path(r.get('checkpoint', '')).name,
            'benchmark' : r.get('benchmark', ''),
            'accuracy'  : f"{r.get('accuracy', 0):.1%}",
            'correct'   : f"{r.get('n_correct', 0)}/{r.get('n_total', 0)}",
            'n_samples' : r.get('n_samples', 1),
        })
    if rows:
        df = (pd.DataFrame(rows)
                .sort_values(['benchmark', 'accuracy'], ascending=[True, False])
                .reset_index(drop=True))
        display(df)
    else:
        print('No results yet — run evaluation cells first.')
    return rows


print('Helpers loaded.')

---
## § 3 — Data

Verify training data and preview AIME problems.  
If a tier is missing, build it with `!python -m code.data_clean.build --tier train`.

In [ ]:
from code.data import load_aime, load_train_corpus

# ── Training corpus stats ──────────────────────────────────────────────────
for tier in ['train_light', 'train']:
    try:
        corpus = load_train_corpus(tier)
        srcs = {}
        for row in corpus:
            s = row.get('source', 'unknown')
            srcs[s] = srcs.get(s, 0) + 1
        print(f'{tier:12s}: {len(corpus):>6,} rows  |  {srcs}')
    except FileNotFoundError:
        print(f'{tier:12s}: NOT BUILT  -> python -m code.data_clean.build --tier {tier}')

print()

# ── AIME preview ──────────────────────────────────────────────────────────
for year in [2024, 2025]:
    problems = load_aime(year)
    p0 = problems[0]
    print(f'AIME {year}: {len(problems)} problems')
    print(f'  [{p0["id"]}]  answer={p0["answer"]}')
    print(f'  {p0["problem"][:120]}...')
    print()

---
## § 4 — Experiments

Each cell is **independent** — run any one without running the others.

| Cell | Experiment | Trains? | Notes |
|------|-----------|---------|-------|
| E3 | Zero-shot Llama | No | Just eval, no adapter |
| E1 | peft LoRA | Yes | Primary comparison baseline |
| E2 | peft DoRA | Yes | Sanity-checks our scratch DoRA |
| E4 | Rank sweep | Yes | Both methods at r ∈ {4,8,16,32,64} |
| E5 | QDoRA (4-bit) | Yes | Enable `load_in_4bit=True` in CFG |

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  E3 — Zero-shot Llama  (B0 accuracy floor, no fine-tuning)  ║
# ╚══════════════════════════════════════════════════════════════╝
run_eval(CFG['model_name'])

In [ ]:
# ╔════════════════════════════════════════════════════════╗
# ║  E1 — peft LoRA  (B1 primary headline comparison)     ║
# ╚════════════════════════════════════════════════════════╝
run_train('lora')
run_eval(latest_checkpoint('lora_'))

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  E2 — peft DoRA  (B2 sanity-check for our scratch DoRA)     ║
# ╚══════════════════════════════════════════════════════════════╝
run_train('peft_dora')
run_eval(latest_checkpoint('peft_dora_'))

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  E4 — Rank sweep  (reproduces paper Fig 5)                             ║
# ║  LoRA + peft_DoRA at r ∈ {4, 8, 16, 32, 64}, always alpha = 2 * rank  ║
# ║  Switch 'peft_dora' -> 'dora' once scratch DoRA is implemented.        ║
# ╚══════════════════════════════════════════════════════════════════════════╝
SWEEP_RANKS = [4, 8, 16, 32, 64]

for r in SWEEP_RANKS:
    print(f'\n=== LoRA rank {r} ===')
    run_train('lora',      overrides={'rank': r, 'alpha': 2 * r})

for r in SWEEP_RANKS:
    print(f'\n=== peft_dora rank {r} ===')
    run_train('peft_dora', overrides={'rank': r, 'alpha': 2 * r})

# Eval all rank-sweep checkpoints on AIME 2024
for ckpt in list_checkpoints():
    rc_path = os.path.join(ckpt, 'run_config.json')
    if os.path.exists(rc_path):
        rc = json.load(open(rc_path))
        if rc.get('rank') in SWEEP_RANKS:
            run_eval(ckpt, benchmarks='aime2024')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  Scratch DoRA  (our re-implementation — E-main headline result) ║
# ╚══════════════════════════════════════════════════════════════════╝
run_train('dora')
run_eval(latest_checkpoint('dora_'))

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  Scratch DoRA  (E1 main — uncomment once dora_layers is done)  ║
# ╚══════════════════════════════════════════════════════════════════╝
# run_train('dora')
# run_eval(latest_checkpoint('dora_'))

---
## § 5 — Results

In [ ]:
# All experiments side-by-side
show_results()

In [ ]:
# Checkpoint inventory
list_checkpoints()

---
## § 6 — One-off utilities

Cells for common ad-hoc tasks.

In [ ]:
# ── Eval a specific checkpoint manually ──────────────────────────────────
# ckpt = 'results/checkpoints/lora_train_MMDD_HHMM'
# run_eval(ckpt, benchmarks='aime2025', n_samples=16, temperature=0.7)

In [ ]:
# ── Build training data ─────────────────────────────────────────────────
# !python -m code.data_clean.build --tier train_light
# !python -m code.data_clean.build --tier train
# !python -m code.data_clean.build --tier train_scale  # large, downloads OMI-2

In [ ]:
# ── LR sweep helper ──────────────────────────────────────────────────────
# Try a few LRs on train_light before committing to a full train run.
# for lr in [5e-5, 1e-4, 2e-4]:
#     run_train('lora', tier='train_light', overrides={'lr': lr, 'epochs': 1})
# show_results()